# **Aprendizaje por refuerzos** - FrozenLake y LunarLander (Gymnasium)

## Tarea: Implementar Agentes Q-Learning y DQN

### Objetivos:
1. Implementar el algoritmo Q-Learning
2. Implementar el algoritmo DQN
3. Entrenar y evaluar ambos agentes
4. Comparar el rendimiento de ambos enfoques


In [1]:
# Instalar paquetes requeridos
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

%pip install swig matplotlib gymnasium torch pygame


Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importar las bibliotecas
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import pygame
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from collections import deque, namedtuple
import random


/home/hezek/.pyenv/versions/3.13.6/lib/python3.13/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


La siguiente celda permite ejecutar un juego de Frozen Lake *determinista* para jugar con el teclado.

Utilize las teclas de dirección (flechas) o asdw para comandar al agente.


In [3]:
def jugar_frozen_lake(env):
    env.reset()
    
    print("Controles:")
    print("W - Arriba")
    print("S - Abajo") 
    print("A - Izquierda")
    print("D - Derecha")
    print("Q - Salir")
    print("Presione cualquier tecla para empezar...")
    
    pygame.init()
    pygame.display.set_caption("FrozenLake - Juego Interactivo")
    
    clock = pygame.time.Clock()
    ejecutando = True
    
    while ejecutando:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                ejecutando = False
            elif event.type == pygame.KEYDOWN:
                if event.key == pygame.K_q or event.key == pygame.K_ESCAPE:
                    ejecutando = False
                elif event.key == pygame.K_w or event.key == pygame.K_UP:
                    accion = 3  # Arriba
                elif event.key == pygame.K_s or event.key == pygame.K_DOWN:
                    accion = 1  # Abajo
                elif event.key == pygame.K_a or event.key == pygame.K_LEFT:
                    accion = 0  # Izquierda
                elif event.key == pygame.K_d or event.key == pygame.K_RIGHT:
                    accion = 2  # Derecha
                else:
                    continue
                
                observacion, recompensa, terminado, truncado, info = env.step(accion)
                print(f"Acción: {accion}, Recompensa: {recompensa}, Terminado: {terminado}")
                
                if terminado or truncado:
                    print(f"¡Episodio terminado! Recompensa final: {recompensa}")
                    pygame.time.wait(500)
                    env.reset()
        
        clock.tick(60)
    
    pygame.quit()
    env.close()

env = gym.make('FrozenLake-v1', render_mode='human', is_slippery=False)
# Descomente la línea de abajo para jugar interactivamente
#jugar_frozen_lake(env)

La siguiente celda permite jugar al juego no determinista.

In [4]:
env = gym.make('FrozenLake-v1', map_name="4x4", render_mode='human', is_slippery=True)
# Descomente la línea de abajo para jugar interactivamente
#jugar_frozen_lake(env)

La siguiente clase define la interfaz de los agentes que utilizaremos para jugar al Frozen Lake.


In [5]:
from abc import ABC, abstractmethod

class Agente(ABC):
    
    @abstractmethod
    def elegir_accion(self, estado):
        """Elige una acción dada una observación."""
        pass

    @abstractmethod
    def epsilon_decay_step(self, episodios):
        pass  # El agente aleatorio no usa epsilon

    @abstractmethod
    def aprender(self, estado, accion, recompensa, siguiente_estado, terminado):
        """Aprende de la experiencia."""
        pass


class AgenteAleatorio(Agente):
    """Agente aleatorio que elige acciones al azar."""
    
    def __init__(self, espacio_acciones):
        # Se guarda el espacio de acciones para poder elegir acciones al azar
        self.espacio_acciones = espacio_acciones
    
    def elegir_accion(self, estado):
        return self.espacio_acciones.sample()

    def epsilon_decay_step(self, episodios):
        pass  # El agente aleatorio no usa epsilon

    def aprender(self, estado, accion, recompensa, siguiente_estado, terminado):
        pass  # El agente aleatorio no aprende

# Probar el AgenteAleatorio
env = gym.make('FrozenLake-v1')
agente_aleatorio = AgenteAleatorio(env.action_space)
estado, _ = env.reset()
accion = agente_aleatorio.elegir_accion(estado)
print(f"✓ AgenteAleatorio creado y probado. Acción: {accion}")
env.close()


✓ AgenteAleatorio creado y probado. Acción: 1


La siguiente celda define una función para evaluar el desempeño de un agente dado.

In [6]:
# Función de Evaluación de Agentes
def evaluar_agente(agente, env, num_episodios=1000):
    """
    Evalúa el rendimiento de un agente a lo largo de múltiples episodios.
    
    Args:
        agente: El agente a evaluar
        env: El entorno
        num_episodios: Número de episodios a ejecutar
    
    Returns:
        dict: Resultados de la evaluación
    """
    recompensas_totales = []
    victorias = 0
    
    for episodio in range(num_episodios):
        estado, _ = env.reset()
        recompensa_total = 0
        
        while True:
            accion = agente.elegir_accion(estado)
            estado, recompensa, terminado, truncado, _ = env.step(accion)
            recompensa_total += recompensa
            
            if terminado or truncado:
                break
        
        recompensas_totales.append(recompensa_total)
        if recompensa_total > 0:
            victorias += 1
    
    return {
        'recompensas_totales': recompensas_totales,
        'victorias': victorias,
        'tasa_victorias': victorias / num_episodios,
        'recompensa_promedio': np.mean(recompensas_totales),
        'desv_estandar': np.std(recompensas_totales)
    }

def imprimir_resultados_evaluacion(resultados, nombre_agente):
    """Imprime los resultados de evaluación de forma formateada."""
    print(f"\n{nombre_agente} - Resultados de Evaluación:")
    print(f"Tasa de Victorias: {resultados['tasa_victorias']:.1%}")
    print(f"Recompensa Promedio: {resultados['recompensa_promedio']:.3f}")
    print(f"Desviación Estándar: {resultados['desv_estandar']:.3f}")
    print(f"Total de Victorias: {resultados['victorias']}")

# Probar función de evaluación
env = gym.make('FrozenLake-v1')
agente_aleatorio = AgenteAleatorio(env.action_space)
resultados = evaluar_agente(agente_aleatorio, env, num_episodios=100)
imprimir_resultados_evaluacion(resultados, "Agente Aleatorio")
env.close()



Agente Aleatorio - Resultados de Evaluación:
Tasa de Victorias: 1.0%
Recompensa Promedio: 0.010
Desviación Estándar: 0.099
Total de Victorias: 1


La siguiente celda define una función para entrenar un agente.

In [7]:
# Función de Entrenamiento de Agentes
def entrenar_agente(agente, env, num_episodios=1000, max_pasos=500, verbose=True):
    """
    Entrena un agente en el entorno.
    
    Args:
        agente: El agente a entrenar
        env: El entorno
        num_episodios: Número de episodios de entrenamiento
        max_pasos: Máximo de pasos por episodio
        verbose: Si imprimir el progreso
    
    Returns:
        list: Recompensas de episodios
    """
    recompensas_episodios = []
    longitudes_episodios = []
    num_episodios_10 = int(num_episodios / 10)
    
    for episodio in range(num_episodios):
        estado, _ = env.reset()
        recompensa_total = 0
        pasos = 0
        
        for paso in range(max_pasos):
            accion = agente.elegir_accion(estado)
            siguiente_estado, recompensa, terminado, truncado, _ = env.step(accion)

            agente.aprender(estado, accion, recompensa, siguiente_estado, terminado or truncado)
            
            estado = siguiente_estado
            recompensa_total += recompensa
            pasos += 1
            
            if terminado or truncado:
                agente.epsilon_decay_step(num_episodios)
                break
        
        recompensas_episodios.append(recompensa_total)
        longitudes_episodios.append(pasos)
        
        if verbose and (episodio + 1) % num_episodios_10 == 0:
            recompensa_promedio = np.mean(recompensas_episodios[-num_episodios_10:])
            longitud_promedio = np.mean(longitudes_episodios[-num_episodios_10:])
            print(f"Episodio {episodio + 1}: Recompensa Promedio = {recompensa_promedio:.3f}, Longitud Promedio = {longitud_promedio:.1f}")
    
    return recompensas_episodios, longitudes_episodios

print("✓ Funciones de entrenamiento definidas")



✓ Funciones de entrenamiento definidas


## Q-Learning en entornos no deterministas

En entornos estocásticos, ejecutar una acción desde un estado no garantiza llegar siempre al mismo siguiente estado, ni recibir siempre la misma recompensa. Por eso, la actualización de la tabla Q se hace de manera **promediada**, usando un factor de aprendizaje α.

La fórmula clásica de actualización es:

$$
Q(s, a) \leftarrow (1-\alpha) Q(s, a) + \alpha \big[r + \gamma \max_{a'} Q(s', a')\big]
$$

- **α (tasa de aprendizaje):** determina cuánto peso le damos a la nueva información frente a lo aprendido hasta ahora.  
- **r + γ max Q(s', a'):** estimación de la recompensa total esperada al ejecutar la acción.
- En **entornos deterministas**, se puede sobrescribir Q(s, a) directamente porque la acción siempre lleva al mismo estado y recompensa.
- En **entornos no deterministas**, se necesita promediar la experiencia, ya que la acción puede llevar a distintos estados con distintas recompensas. El valor de Q converge así al **valor esperado a largo plazo** de cada acción.


In [8]:
class AgenteQLearning(Agente):
    """Agente que usa el algoritmo Q-Learning."""
    
    def __init__(self, espacio_estados, espacio_acciones):
        self.Q_table = np.zeros((espacio_estados.n, espacio_acciones.n))
        self.alpha = 0.05  # Tasa de aprendizaje
        self.gamma = 0.95 # Factor de descuento
        self.epsilon = 1 # Tasa de exploración inicial
        self.epsilon_decay = 0.995
        self.epsilon_min = 0
        self.espacio_acciones = espacio_acciones
        self.tam_mapa = int(np.sqrt(espacio_estados.n))  

    def elegir_accion(self, estado):
        """Elige una acción usando política epsilon-greedy.""" #o softmax, lo cambio a softmax luego 
        if np.random.rand() < self.epsilon:
            return np.random.randint(4)  # Exploración
        else:
            return np.argmax(self.Q_table[estado])  # Explotación, accion que maximiza Q
        
    def epsilon_decay_step(self, episodios):
        self.epsilon = max(self.epsilon_min, self.epsilon - 1/episodios)
            
    def aprender(self, estado, accion, recompensa, siguiente_estado, terminado):
        """Actualiza la tabla Q usando la ecuación de Bellman."""
        max_q = np.max(self.Q_table[siguiente_estado])
        self.Q_table[estado, accion] = (1 - self.alpha) * self.Q_table[estado, accion] + self.alpha * (recompensa + self.gamma * max_q)
    
    def imprimirQtable(self):
        print("Tabla Q:")
        print(self.Q_table)

    def mostrar_policy(self, env):
        """
        Muestra la política aprendida en el mapa del FrozenLake.
        Flechas solo en casillas 'F' (Frozen); 'H' y 'G' se muestran tal cual.
        """
        # Mapeo de acciones a flechas: 0=izquierda, 1=abajo, 2=derecha, 3=arriba
        acciones_map = ["←", "↓", "→", "↑"]

        # Obtener el mapa original del entorno
        desc = env.unwrapped.desc.astype(str)

        # Crear copia del mapa para mostrar la política
        policy_map = desc.copy()

        # Recorrer filas y columnas del mapa
        for fila in range(self.tam_mapa):
            for col in range(self.tam_mapa):
                estado = fila * self.tam_mapa + col  # convertir fila,col a índice lineal
                if desc[fila, col] == 'F':  # solo actualizar casillas 'Frozen'
                    mejor_accion = np.argmax(self.Q_table[estado])
                    policy_map[fila, col] = acciones_map[mejor_accion]

        print("Policy aprendida:")
        for fila in policy_map:
            print(" ".join(fila))

    def guardar_qtable(self, path="qtable.npy"):
        np.save(path, self.Q_table)
        print(f"Q-table guardada en {path}")

    def cargar_qtable(self, path="qtable.npy"):
        self.Q_table = np.load(path)
        self.epsilon = self.epsilon_min  # Fijar epsilon al mínimo para evaluación
        print(f"Q-table cargada desde {path}")


### Entrenar Agente QL tabular, Frozen Lake 4x4

In [9]:
# Entrenar y Evaluar el Agente Q-Learning en FrozenLake-v1 4x4 
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True, render_mode="ansi",  max_episode_steps=500)
agente = AgenteQLearning(env.observation_space, env.action_space)
estado, _ = env.reset()
print(f"FrozenLake-v1 4x4: {env.render()}")
resultados, longitudes = entrenar_agente(agente, env, num_episodios=1000000, verbose=True)
agente.imprimirQtable()
agente.guardar_qtable("qtable_frozenlake_4x4.npy")
env.close()

FrozenLake-v1 4x4: 
SFFF
FHFH
FFFH
HFFG

Episodio 100000: Recompensa Promedio = 0.016, Longitud Promedio = 8.0
Episodio 200000: Recompensa Promedio = 0.023, Longitud Promedio = 8.7
Episodio 300000: Recompensa Promedio = 0.031, Longitud Promedio = 9.5
Episodio 400000: Recompensa Promedio = 0.042, Longitud Promedio = 10.5
Episodio 500000: Recompensa Promedio = 0.060, Longitud Promedio = 11.8
Episodio 600000: Recompensa Promedio = 0.084, Longitud Promedio = 13.5
Episodio 700000: Recompensa Promedio = 0.125, Longitud Promedio = 15.7
Episodio 800000: Recompensa Promedio = 0.187, Longitud Promedio = 18.9
Episodio 900000: Recompensa Promedio = 0.301, Longitud Promedio = 24.1
Episodio 1000000: Recompensa Promedio = 0.536, Longitud Promedio = 34.0
Tabla Q:
[[0.15389307 0.14609218 0.14731944 0.14369449]
 [0.11537745 0.08123379 0.09441178 0.14891852]
 [0.13166253 0.11907876 0.11536236 0.11378018]
 [0.08660184 0.06618333 0.07649691 0.10896038]
 [0.17282719 0.12610533 0.14866076 0.12494432]
 [0.   

### Evaluar Agente QL tabular, Frozen Lake 4x4

In [10]:
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True, render_mode="ansi",  max_episode_steps=500)
agente = AgenteQLearning(env.observation_space, env.action_space)
estado, _ = env.reset()
print(f"FrozenLake-v1 4x4: {env.render()}")
agente.cargar_qtable("qtable_frozenlake_4x4.npy")
resultados = evaluar_agente(agente, env, num_episodios=1000)
imprimir_resultados_evaluacion(resultados, "Agente Q-Learning")
agente.mostrar_policy(env)
env.close()

FrozenLake-v1 4x4: 
SFFF
FHFH
FFFH
HFFG

Q-table cargada desde qtable_frozenlake_4x4.npy

Agente Q-Learning - Resultados de Evaluación:
Tasa de Victorias: 80.5%
Recompensa Promedio: 0.805
Desviación Estándar: 0.396
Total de Victorias: 805
Policy aprendida:
S ↑ ← ↑
← H ← H
↑ ↓ ← H
H → ↓ G


### Entrenar Agente QL tabular, Frozen Lake 8x8

In [11]:
# Entrenar  Agente Q-Learning en FrozenLake-v1 8x8
env = gym.make("FrozenLake-v1", map_name="8x8", is_slippery=True, render_mode="ansi",  max_episode_steps=500)
agente = AgenteQLearning(env.observation_space, env.action_space)
estado, _ = env.reset()
print(f"FrozenLake-v1 8x8: {env.render()}")
resultados, longitudes = entrenar_agente(agente, env, num_episodios=100000, verbose=True)
agente.imprimirQtable()
agente.guardar_qtable("qtable_frozenlake_8x8.npy")
env.close()

FrozenLake-v1 8x8: 
SFFFFFFF
FFFFFFFF
FFFHFFFF
FFFFFHFF
FFFHFFFF
FHHFFFHF
FHFFHFHF
FFFHFFFG

Episodio 10000: Recompensa Promedio = 0.004, Longitud Promedio = 33.2
Episodio 20000: Recompensa Promedio = 0.007, Longitud Promedio = 36.7
Episodio 30000: Recompensa Promedio = 0.010, Longitud Promedio = 39.1
Episodio 40000: Recompensa Promedio = 0.022, Longitud Promedio = 42.6
Episodio 50000: Recompensa Promedio = 0.039, Longitud Promedio = 46.0
Episodio 60000: Recompensa Promedio = 0.073, Longitud Promedio = 50.0
Episodio 70000: Recompensa Promedio = 0.115, Longitud Promedio = 53.5
Episodio 80000: Recompensa Promedio = 0.197, Longitud Promedio = 57.8
Episodio 90000: Recompensa Promedio = 0.336, Longitud Promedio = 62.3
Episodio 100000: Recompensa Promedio = 0.558, Longitud Promedio = 70.8
Tabla Q:
[[3.91475602e-02 3.96054605e-02 4.01317462e-02 4.52767276e-02]
 [4.42203967e-02 4.41351832e-02 5.18065109e-02 4.44934135e-02]
 [5.43888113e-02 5.59086801e-02 6.06812610e-02 5.44993362e-02]
 [6.6435

### Evaluar Agente QL tabular, Frozen Lake 8x8

In [12]:
env = gym.make("FrozenLake-v1", map_name="8x8", is_slippery=True, render_mode="ansi",  max_episode_steps=500)
agente = AgenteQLearning(env.observation_space, env.action_space)
estado, _ = env.reset()
print(f"FrozenLake-v1 8x8: {env.render()}")
agente.cargar_qtable("qtable_frozenlake_8x8.npy")
resultados = evaluar_agente(agente, env, num_episodios=1000)
imprimir_resultados_evaluacion(resultados, "Agente Q-Learning")
agente.mostrar_policy(env)
env.close()

FrozenLake-v1 8x8: 
SFFFFFFF
FFFFFFFF
FFFHFFFF
FFFFFHFF
FFFHFFFF
FHHFFFHF
FHFFHFHF
FFFHFFFG

Q-table cargada desde qtable_frozenlake_8x8.npy

Agente Q-Learning - Resultados de Evaluación:
Tasa de Victorias: 85.0%
Recompensa Promedio: 0.850
Desviación Estándar: 0.357
Total de Victorias: 850
Policy aprendida:
S → → → → → → →
↑ ↑ ↑ ↑ → → → ↓
↑ ↑ ← H → ↑ → ↓
↑ ↑ ↑ ↓ ← H → →
← ↑ ↑ H → ↓ ↑ →
← H H → ↑ ← H →
← H → ↑ H ← H →
← ↓ ↑ H → ↓ ↓ G


La siguiente celda define el agente DQN a implementar. 

In [ ]:

class DQN(nn.Module):
    """Clase auxiliar que implementa una Red Q Profunda con una capa oculta."""
    
    def __init__(self, tamano_entrada, tamano_oculto, tamano_salida):
        super(DQN, self).__init__()
        
        self.fc1 = nn.Linear(tamano_entrada, tamano_oculto)
        self.out = nn.Linear(tamano_oculto, tamano_salida)
    
    def forward(self, x):
        x = F.relu(self.fc1(x)) #Func. activación ReLU (Rectified Linear Unit)
        x = self.out(x)         #Capa final no usa activación
        return x                

class AgenteDQN(Agente):
    """Agente de Red Q Profunda."""
    
    def __init__(self,
                 espacio_estados,
                 espacio_acciones,
                 tamano_oculto=64,
                 lr=1e-3,
                 gamma=0.99,
                 epsilon=1.0,
                 epsilon_decay=0.995,
                 epsilon_min=0.01,
                 replay_size=10000,
                 batch_size=64,
                 target_update=1000,
                 train_freq=4,
                 device=None):
        """
        Args:
            espacio_estados: env.observation_space (discreto)
            espacio_acciones: env.action_space
            tamano_oculto: neuronas en la capa oculta
            lr: learning rate para el optimizador
            gamma: factor de descuento
            epsilon, epsilon_decay, epsilon_min: para epsilon-greedy
            replay_size: capacidad del buffer de replay
            batch_size: tamaño de batch para entrenar
            target_update: cada cuántos pasos copiar la target network
            train_freq: entrenar cada 'train_freq' llamadas a aprender
            device: 'cpu' o 'cuda' (None -> detecta automaticamente)
        """
        # parámetros del entorno
        self.num_states = espacio_estados.n
        self.num_actions = espacio_acciones.n
        
        # hiperparámetros
        self.tamano_oculto = tamano_oculto
        self.lr = lr
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.epsilon_min = epsilon_min
        self.batch_size = batch_size
        self.target_update = target_update
        self.train_freq = train_freq
        
        # device
        if device is None:
            self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        else:
            self.device = torch.device(device)
        
        # redes: policy_net y target_net
        self.policy_net = DQN(self.num_states, self.tamano_oculto, self.num_actions).to(self.device)
        self.target_net = DQN(self.num_states, self.tamano_oculto, self.num_actions).to(self.device)
        
        # inicializar target igual a policy
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()
        
        # optimizador y loss
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=self.lr)
        self.loss_fn = nn.MSELoss()
        
        # replay buffer
        self.replay_buffer = deque(maxlen=replay_size)
        
        # contadores
        self.step_count = 0  # para updates y frecuencia de entrenamiento
        
        # semilla opcional
        random.seed(0)
        np.random.seed(0)
        torch.manual_seed(0)
    
    def _one_hot(self, estado):
        """Convierte un entero estado -> vector one-hot tensor en device."""
        vec = np.zeros(self.num_states, dtype=np.float32)
        vec[estado] = 1.0
        return torch.from_numpy(vec).to(self.device)
    
    def elegir_accion(self, estado):
        """Elige acción usando política epsilon-greedy."""
        # exploración
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.num_actions)
        
        # explotación: predecir Q-values y tomar argmax
        self.policy_net.eval()
        with torch.no_grad():
            s_t = self._one_hot(estado).unsqueeze(0)  # shape (1, num_states)
            q_vals = self.policy_net(s_t)            # shape (1, num_actions)
            accion = torch.argmax(q_vals, dim=1).item()
        self.policy_net.train()
        return accion
    
    def aprender(self, estado, accion, recompensa, siguiente_estado, terminado):
        """
        Agrega la transición al buffer y entrena la red periódicamente.
        Este método es llamado por el loop de entrenamiento por paso.
        """
        # almacenar transición (s, a, r, s', done)
        self.replay_buffer.append((estado, accion, recompensa, siguiente_estado, float(terminado)))
        self.step_count += 1
        
        # Entrenar cada 'train_freq' pasos (evita entrenar en cada transición)
        if (self.step_count % self.train_freq == 0) and (len(self.replay_buffer) >= self.batch_size):
            self._train_on_batch()
        
        # actualizar target network periódicamente
        if self.step_count % self.target_update == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())
        
        # si terminó el episodio, actualizar epsilon (igual que en Q-table)
        #if terminado:
        #    self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
    
    def epsilon_decay_step(self, episodios):
        self.epsilon = max(self.epsilon_min, self.epsilon - 1/episodios)

    def _train_on_batch(self):
        """Muestrea un minibatch y realiza un paso de optimización."""
        batch = random.sample(self.replay_buffer, self.batch_size)
        estados, acciones, recompensas, estados_siguientes, dones = zip(*batch)
        
        # convertir a tensores
        estados = torch.stack([self._one_hot(s) for s in estados])            # (B, num_states)
        estados_siguientes = torch.stack([self._one_hot(s) for s in estados_siguientes])  # (B, num_states)
        acciones = torch.tensor(acciones, dtype=torch.long, device=self.device)           # (B,)
        recompensas = torch.tensor(recompensas, dtype=torch.float32, device=self.device)  # (B,)
        dones = torch.tensor(dones, dtype=torch.float32, device=self.device)              # (B,)
        
        # Q(s,a) predichos por policy_net
        q_preds = self.policy_net(estados)                       # (B, num_actions)
        q_pred_values = q_preds.gather(1, acciones.unsqueeze(1)).squeeze(1)  # (B,)
        
        # target: r + gamma * max_a' Q_target(s', a')  (0 si done)
        with torch.no_grad():
            q_next = self.target_net(estados_siguientes)        # (B, num_actions)
            q_next_max, _ = q_next.max(dim=1)                  # (B,)
            q_targets = recompensas + (1.0 - dones) * (self.gamma * q_next_max)
        
        # loss y optimización
        loss = self.loss_fn(q_pred_values, q_targets)
        self.optimizer.zero_grad()
        loss.backward()
        # opcional: gradient clipping
        torch.nn.utils.clip_grad_norm_(self.policy_net.parameters(), 10.0)
        self.optimizer.step()

    def mostrar_policy(self, env):
        acciones_map = ["←", "↓", "→", "↑"]
        desc = env.unwrapped.desc.astype(str)
        tam_mapa = int(np.sqrt(self.num_states))
        policy_map = desc.copy()
        
        self.policy_net.eval()
        with torch.no_grad():
            all_states = torch.eye(self.num_states, device=self.device)
            q_vals = self.policy_net(all_states)
            mejores = torch.argmax(q_vals, dim=1).cpu().numpy()
        self.policy_net.train()
        
        for fila in range(tam_mapa):
            for col in range(tam_mapa):
                estado = fila * tam_mapa + col
                if desc[fila, col] == 'F':
                    policy_map[fila, col] = acciones_map[mejores[estado]]
        
        print("Policy aprendida (DQN):")
        for fila in policy_map:
            print(" ".join(fila))    

    def guardar_policy(self, path="dqn_policy.pth"):
        torch.save(self.policy_net.state_dict(), path)
        print(f"Policy guardada en {path}")

    def cargar_policy(self, path="dqn_policy.pth"):
        self.policy_net.load_state_dict(torch.load(path, map_location=self.device))
        self.policy_net.eval()
        self.epsilon = self.epsilon_min  # Fijar epsilon al mínimo para evaluación
        print(f"Policy cargada desde {path}")

### Entrenar Agente DQN, Frozen Lake 4x4


In [14]:
# Entrenar y Evaluar el Agente DQN en FrozenLake-v1 4x4 
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True, render_mode="ansi",  max_episode_steps=500)
agente = AgenteDQN(env.observation_space, env.action_space, tamano_oculto=32)
estado, _ = env.reset()
print(f"FrozenLake-v1 4x4: {env.render()}")
resultados, longitudes = entrenar_agente(agente, env, num_episodios=20000, verbose=True)
agente.guardar_policy("dqn_policy_frozenlake_4x4.pth")
env.close()

FrozenLake-v1 4x4: 
SFFF
FHFH
FFFH
HFFG

Episodio 2000: Recompensa Promedio = 0.018, Longitud Promedio = 8.1
Episodio 4000: Recompensa Promedio = 0.021, Longitud Promedio = 8.2
Episodio 6000: Recompensa Promedio = 0.032, Longitud Promedio = 9.7
Episodio 8000: Recompensa Promedio = 0.047, Longitud Promedio = 11.0
Episodio 10000: Recompensa Promedio = 0.051, Longitud Promedio = 12.3
Episodio 12000: Recompensa Promedio = 0.076, Longitud Promedio = 14.1
Episodio 14000: Recompensa Promedio = 0.116, Longitud Promedio = 16.8
Episodio 16000: Recompensa Promedio = 0.180, Longitud Promedio = 20.7
Episodio 18000: Recompensa Promedio = 0.300, Longitud Promedio = 27.6
Episodio 20000: Recompensa Promedio = 0.582, Longitud Promedio = 41.8
Policy guardada en dqn_policy_frozenlake_4x4.pth


### Evaluar Agente DQN, Frozen Lake 4x4

In [15]:
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True, render_mode="ansi",  max_episode_steps=500)
agente = AgenteDQN(env.observation_space, env.action_space, tamano_oculto=32)
agente.cargar_policy("dqn_policy_frozenlake_4x4.pth")
estado, _ = env.reset()
resultados = evaluar_agente(agente, env, num_episodios=1000)
imprimir_resultados_evaluacion(resultados, "Agente  DQN")
agente.mostrar_policy(env)
env.close()

Policy cargada desde dqn_policy_frozenlake_4x4.pth

Agente  DQN - Resultados de Evaluación:
Tasa de Victorias: 74.3%
Recompensa Promedio: 0.743
Desviación Estándar: 0.437
Total de Victorias: 743
Policy aprendida (DQN):
S ↑ ↑ ↑
← H → H
↑ ↓ ← H
H → ↓ G


### Entrenar Agente DQN, Frozen Lake 8x8

In [16]:
# Entrenar y Evaluar el Agente DQN en FrozenLake-v1 8x8
env = gym.make("FrozenLake-v1", map_name="8x8", is_slippery=True, render_mode="ansi",  max_episode_steps=500)
agente = AgenteDQN(env.observation_space, env.action_space, tamano_oculto=32)
estado, _ = env.reset()
print(f"FrozenLake-v1 8x8: {env.render()}")
resultados, longitudes = entrenar_agente(agente, env, num_episodios=20000, verbose=True)
agente.guardar_policy("dqn_policy_frozenlake_8x8.pth")
env.close()

FrozenLake-v1 8x8: 
SFFFFFFF
FFFFFFFF
FFFHFFFF
FFFFFHFF
FFFHFFFF
FHHFFFHF
FHFFHFHF
FFFHFFFG

Episodio 2000: Recompensa Promedio = 0.001, Longitud Promedio = 33.4
Episodio 4000: Recompensa Promedio = 0.002, Longitud Promedio = 37.7
Episodio 6000: Recompensa Promedio = 0.011, Longitud Promedio = 40.7
Episodio 8000: Recompensa Promedio = 0.018, Longitud Promedio = 47.2
Episodio 10000: Recompensa Promedio = 0.042, Longitud Promedio = 56.3
Episodio 12000: Recompensa Promedio = 0.075, Longitud Promedio = 61.2
Episodio 14000: Recompensa Promedio = 0.123, Longitud Promedio = 73.3
Episodio 16000: Recompensa Promedio = 0.236, Longitud Promedio = 82.9
Episodio 18000: Recompensa Promedio = 0.414, Longitud Promedio = 99.5
Episodio 20000: Recompensa Promedio = 0.676, Longitud Promedio = 105.7
Policy guardada en dqn_policy_frozenlake_8x8.pth


### Evaluar Agente DQN, Frozen Lake 8x8

In [17]:
env = gym.make("FrozenLake-v1", map_name="8x8", is_slippery=True, render_mode="ansi",  max_episode_steps=500)
agente = AgenteDQN(env.observation_space, env.action_space, tamano_oculto=32)
agente.cargar_policy("dqn_policy_frozenlake_8x8.pth")
estado, _ = env.reset()
resultados = evaluar_agente(agente, env, num_episodios=1000)
imprimir_resultados_evaluacion(resultados, "Agente  DQN")
agente.mostrar_policy(env)
env.close()

Policy cargada desde dqn_policy_frozenlake_8x8.pth

Agente  DQN - Resultados de Evaluación:
Tasa de Victorias: 82.9%
Recompensa Promedio: 0.829
Desviación Estándar: 0.377
Total de Victorias: 829
Policy aprendida (DQN):
S ↓ → ↓ → → → ↓
↑ ↑ ↑ ↑ ↑ ↑ → ↓
← ← ← H → → → ↓
↑ ↑ ↑ ↑ → H → →
↑ ↑ ↑ H ↑ ↓ ↑ →
→ H H ↑ ↑ ↑ H →
← H ↑ → H ↑ H →
→ ↑ ↑ H ↓ → ↓ G
